In [ ]:
from cgra import *
from kernels import *

In [ ]:
kernel_name = "benchmarks/mmul/5x5"
version = "_gen"

In [ ]:
# Global variables
CGRA_N_ROWS = 5
CGRA_N_COLS = 5
# Adress
first_addr = 20000

In [ ]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [ ]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &B[0][0]          &C[0][1]        &A[0][0]        nRowsBlocksC    nColsBlocksC
    # nColsBlocksC      &B[0][1]        &C[1][2]        &A[1][0]        -
    # -                 loopColsA       &B[0][2]        &C[2][3]        &A[2][0]
    # &A[3][0]          -               nColsBlocksC    &B[0][3]        &C[3][4]
    # &C[4][0]          &A[4][0]        -               -               &B[0][4]
    # ----------------------
    # -4*colsB          colsA           -               -               -
    # -                 -4*colsB        colsA           -               -
    # -                 -               -4*colsB        colsA           -
    # -                 -               -               -4*colsB        colsA
    # colsA             -               -               -               -4*colsB
    nItLoopColsA = colsA
    nColsBlocksC = int(colsB/CGRA_N_ROWS)
    nRowsBlocksC = int(rowsA/CGRA_N_ROWS)
    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_C = first_addr_B + colsA*colsB*4

    config_vals_col0 = [first_addr_B, nColsBlocksC, first_addr_A + 3*colsA*4, first_addr_C + 4*colsB*4, -4*colsB, colsA]
    config_vals_col1 = [first_addr_C + 4, first_addr_B + 4, nItLoopColsA, first_addr_A + 4*colsA*4, colsA, -4*colsB]
    config_vals_col2 = [first_addr_A, first_addr_C + 2*4 + colsB*4, first_addr_B + 2*4, nColsBlocksC, colsA, -4*colsB]
    config_vals_col3 = [nRowsBlocksC, first_addr_A + colsA*4, first_addr_C + 3*4 + 2*colsB*4, first_addr_B + 3*4, colsA, -4*colsB]
    config_vals_col4 = [nColsBlocksC, first_addr_A + 2*colsA*4, first_addr_C + 4*4 + 3*colsB*4, first_addr_B + 4*4, colsA, -4*colsB]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    addr_config_loads_col4 = addr_config_loads_col3 + len(config_vals_col3)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col4, config_vals_col4, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3, addr_config_loads_col4]
    return load_addrs

In [ ]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [ ]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [ ]:
def mmul_cpu(A_data, B_data, C_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum 
    return expected_res

In [ ]:
# Test dimensions (5xXx5)
rowsA = 30
colsA = 23
colsB = 35
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
C_data = [x + 200 for x in range(0, rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB)

In [ ]:
runKernel(load_addrs, max_it=20000)

In [ ]:
# Get result from CGRA
first_addr_C = first_addr + rowsA*colsA*4 + colsA*colsB*4
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)
# Process estra rows/cols
if rowsA%5 != 0:
    for rA in range(rowsA - rowsA%4, rowsA):
        for cB in range(colsB):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum 
if colsB%5 != 0:
    for cB in range(colsB - colsB%4, colsB):
        for rA in range(rowsA - rowsA%4):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum

# Get cpu output
expected_res = mmul_cpu(A_data_cpy, B_data_cpy, C_data_cpy, rowsA, colsA, colsB)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")

